In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

featured = pd.read_csv("../data/processed/airbnb_featured.csv")

print(featured.shape)
featured.head()

(96871, 93)


,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_text_count,first_review_date,last_review_date,avg_review_length,neighbourhood_group,neighbourhood_y,revenue_category,price_category,host_experience_years,review_lifetime_days
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,55.0,2010-08-18,2025-08-21,234.909091,NaN,Islington,Low,Budget,16.602740,5482.0
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,97.0,2009-12-21,2025-04-05,426.371134,NaN,Kensington and Chelsea,Low,Standard,16.550685,5584.0
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,city scrape,Very Central Modern 3-Bed/2 Bath By Oxford St W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,56.0,2011-03-21,2024-02-19,379.892857,NaN,Westminster,No Revenue,Premium,16.468493,4718.0
3,24328,https://www.airbnb.com/rooms/24328,20250914034649,2025-09-18,previous scrape,Battersea live/work artist house,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,95.0,2010-11-15,2025-07-05,395.715789,NaN,Wandsworth,NaN,NaN,16.736986,5346.0
4,36274,https://www.airbnb.com/rooms/36274,20250914034649,2025-09-15,city scrape,Bright 1 bedroom apt off brick lane in Shoreditch,*Update June '25- Pump Installed to improve wa...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,133271,...,15.0,2011-03-20,2025-09-06,272.266667,NaN,Tower Hamlets,Medium,Standard,16.076712,5284.0


In [3]:
model_df = featured[
    [
        "estimated_revenue_l365d",
        "price",
        "bedrooms",
        "bathrooms",
        "accommodates",
        "review_scores_rating",
        "occupancy_rate",
        "number_of_reviews",
        "review_text_count",
        "avg_review_length",
        "host_experience_years",
        "room_type",
        "neighbourhood_cleansed",
        "host_is_superhost"
    ]
].copy()

model_df = model_df.dropna(
    subset=["estimated_revenue_l365d"]
)

print(model_df.shape)

model_df.isnull().sum()

(61963, 14)


estimated_revenue_l365d        0
price                          0
bedrooms                     114
bathrooms                     76
accommodates                   0
review_scores_rating       13908
occupancy_rate                 0
number_of_reviews              0
review_text_count          13908
avg_review_length          13908
host_experience_years         25
room_type                      0
neighbourhood_cleansed         0
host_is_superhost           1352
dtype: int64

In [4]:
X = model_df.drop(
    columns=["estimated_revenue_l365d"]
)

y = model_df["estimated_revenue_l365d"]

print(X.shape)
print(y.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(61963, 13)
(61963,)
(49570, 13)
(12393, 13)


In [5]:
numeric_features = [
    "price",
    "bedrooms",
    "bathrooms",
    "accommodates",
    "review_scores_rating",
    "occupancy_rate",
    "number_of_reviews",
    "review_text_count",
    "avg_review_length",
    "host_experience_years"
]

categorical_features = [
    "room_type",
    "neighbourhood_cleansed",
    "host_is_superhost"
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

print("Preprocessor created successfully")

Preprocessor created successfully


In [6]:
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

print("Linear Regression model trained successfully")

Linear Regression model trained successfully


Evaluate Linear Regression

In [7]:
linear_mae = mean_absolute_error(
    y_test,
    linear_predictions
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_predictions
    )
)

linear_r2 = r2_score(
    y_test,
    linear_predictions
)

print("MAE :", linear_mae)
print("RMSE:", linear_rmse)
print("R2  :", linear_r2)

MAE : 10055.51284033968
RMSE: 93813.97887642407
R2  : 0.021121584882438316


In [8]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_model.fit(
    X_train,
    y_train
)

rf_predictions = rf_model.predict(
    X_test
)

print("Random Forest trained successfully")

Random Forest trained successfully


In [9]:
rf_mae = mean_absolute_error(
    y_test,
    rf_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        rf_predictions
    )
)

rf_r2 = r2_score(
    y_test,
    rf_predictions
)

print("MAE :", rf_mae)
print("RMSE:", rf_rmse)
print("R2  :", rf_r2)

MAE : 5974.087640603566
RMSE: 86481.4715159145
R2  : 0.16816003485821363


In [10]:
feature_names = (
    rf_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

importances = (
    rf_model.named_steps["model"]
    .feature_importances_
)

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

importance_df.head(15)

,Feature,Importance
0,num__price,0.467136
8,num__avg_review_length,0.212412
6,num__number_of_reviews,0.092413
4,num__review_scores_rating,0.052599
7,num__review_text_count,0.052420
5,num__occupancy_rate,0.041528
9,num__host_experience_years,0.017996
35,cat__neighbourhood_cleansed_Lambeth,0.015845
3,num__accommodates,0.009733
1,num__bedrooms,0.006972


price model dataset

In [11]:
price_model_df = featured[
    [
        "price",
        "bedrooms",
        "bathrooms",
        "accommodates",
        "review_scores_rating",
        "occupancy_rate",
        "number_of_reviews",
        "review_text_count",
        "avg_review_length",
        "host_experience_years",
        "room_type",
        "neighbourhood_cleansed",
        "host_is_superhost"
    ]
].copy()

# Remove missing target values
price_model_df = price_model_df.dropna(subset=["price"])

# Remove extreme price outliers using 99th percentile
price_cap = price_model_df["price"].quantile(0.99)

price_model_df = price_model_df[
    price_model_df["price"] <= price_cap
]

print(price_model_df.shape)
print("Price cap:", price_cap)

price_model_df.isnull().sum()

(61350, 13)
Price cap: 1100.0


price                         0
bedrooms                    106
bathrooms                    76
accommodates                  0
review_scores_rating      13577
occupancy_rate                0
number_of_reviews             0
review_text_count         13577
avg_review_length         13577
host_experience_years        24
room_type                     0
neighbourhood_cleansed        0
host_is_superhost          1330
dtype: int64

In [12]:
X_price = price_model_df.drop(
    columns=["price"]
)

y_price = price_model_df["price"]

X_train_price, X_test_price, y_train_price, y_test_price = train_test_split(
    X_price,
    y_price,
    test_size=0.2,
    random_state=42
)

print(X_train_price.shape)
print(X_test_price.shape)

(49080, 12)
(12270, 12)


In [13]:
numeric_features_price = [
    "bedrooms",
    "bathrooms",
    "accommodates",
    "review_scores_rating",
    "occupancy_rate",
    "number_of_reviews",
    "review_text_count",
    "avg_review_length",
    "host_experience_years"
]

categorical_features_price = [
    "room_type",
    "neighbourhood_cleansed",
    "host_is_superhost"
]

numeric_transformer_price = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer_price = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor_price = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_price, numeric_features_price),
        ("cat", categorical_transformer_price, categorical_features_price)
    ]
)

print("Price model preprocessor created successfully")

Price model preprocessor created successfully


In [14]:
price_rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_price),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

price_rf_model.fit(
    X_train_price,
    y_train_price
)

price_predictions = price_rf_model.predict(
    X_test_price
)

print("Price Recommendation model trained successfully")

Price Recommendation model trained successfully


In [15]:
price_rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor_price),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

price_rf_model.fit(
    X_train_price,
    y_train_price
)

price_predictions = price_rf_model.predict(
    X_test_price
)

print("Price Recommendation model trained successfully")

Price Recommendation model trained successfully


In [16]:
price_mae = mean_absolute_error(
    y_test_price,
    price_predictions
)

price_rmse = np.sqrt(
    mean_squared_error(
        y_test_price,
        price_predictions
    )
)

price_r2 = r2_score(
    y_test_price,
    price_predictions
)

print("MAE :", price_mae)
print("RMSE:", price_rmse)
print("R2  :", price_r2)

MAE : 55.70405250123791
RMSE: 99.44859835383056
R2  : 0.5780999564465736


In [17]:
feature_names_price = (
    price_rf_model.named_steps["preprocessor"]
    .get_feature_names_out()
)

importances_price = (
    price_rf_model.named_steps["model"]
    .feature_importances_
)

importance_df_price = pd.DataFrame(
    {
        "Feature": feature_names_price,
        "Importance": importances_price
    }
)

importance_df_price = importance_df_price.sort_values(
    by="Importance",
    ascending=False
)

importance_df_price.head(15)

,Feature,Importance
0,num__bedrooms,0.247441
8,num__host_experience_years,0.123904
1,num__bathrooms,0.119015
4,num__occupancy_rate,0.105618
7,num__avg_review_length,0.052904
45,cat__neighbourhood_cleansed_Westminster,0.048040
2,num__accommodates,0.039626
3,num__review_scores_rating,0.036231
32,cat__neighbourhood_cleansed_Kensington and Che...,0.034008
5,num__number_of_reviews,0.032434


In [18]:
import joblib

joblib.dump(
    price_rf_model,
    "../models/price_recommender.pkl"
)

print("Price model saved successfully")

Price model saved successfully


In [19]:
rf_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](13,)","['price','bedrooms','bathrooms',...,'room_type','neighbourhood_cleansed', 'host_is_superhost']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through.

In [20]:
import joblib

joblib.dump(
    rf_model,
    "../models/revenue_predictor.pkl"
)

print("Revenue model saved successfully")

Revenue model saved successfully
